# Library

In [1]:
# SageMath を想定
from sage.all import ZZ, QQ, vector, matrix
from functools import reduce

#import sys; sys.path.append("../modules")
#from Ideal_elm_pair_class import *

%display latex

# Classes

### "Mapping class" Class

In [2]:
class fundamentalGroup:
    # 階数4の自由群を定義し、生成元 (x, y, z, w) を取得
    F = FreeGroup(4, 'x,y,z,w')
    x, y, z, w = F.generators()
    
    # 逆元の定義（~ 演算子）
    X, Y, Z, W = ~x, ~y, ~z, ~w
    BASIS = (x, y, z, w)

    @classmethod
    def comm(cls, u,v):
        if u.parent() != cls.F or v.parent() != cls.F:
            raise TypeError("u and v must be elements of fg.F")
        return u * v * (~u) * (~v)

fg = fundamentalGroup

def conj(cls, u,v):
    # fg.conj(u, v) = v^{^1} u v
    if u.parent() != cls.F or v.parent() != cls.F:
        raise TypeError("u and v must be elements of fg.F")
    return v * u * (~v)
    

# 境界語を定義
fg.bnd = fg.comm(fg.x, fg.y) * fg.comm(fg.z, fg.w)

class DehnTwist(fg):
    # 生成元インデックスと文字表記のマッピング
    GEN_NAMES = {1: 'x', 2: 'y', 3: 'z', 4: 'w'}
        
    x, y, z, w, X, Y, Z, W, bnd = (fg.x, fg.y, fg.z, fg.w, fg.X, fg.Y, fg.Z, fg.W, fg.bnd)
    a_w, b_w, c_w, d_w, e_w, f_w = x, Y * fg.conj(w, z), z, w, bnd * w, y
    ACTION = {'a': {'y': y * a_w}, \
              'A': {'y': y * ~a_w}, \
              'b': {'x': x * b_w, 'y': fg.conj(y, ~b_w), 'z': ~b_w * z}, \
              'B': {'x': x * ~b_w, 'y': fg.conj(y, b_w), 'z': b_w * z}, \
              'c': {'w': w * c_w}, \
              'C': {'w': w * ~c_w}, \
              'd': {'z': z * ~d_w}, \
              'D': {'z': z * d_w}, \
              'e': {'x': fg.conj(x, ~e_w), 'y': fg.conj(y, ~e_w), 'z': ~e_w * z}, \
              'E': {'x': fg.conj(x, e_w), 'y': fg.conj(y, e_w), 'z': e_w * z}, \
              'f': {'x': x * ~f_w}, \
              'F': {'x': x * f_w}}
              
    def __init__(self, twisting_loop: str):
        if twisting_loop not in self.ACTION:
            raise ValueError(f"Unknown Dehn twist generator: {twisting_loop}")
        self.loop = twisting_loop
        self.act = self.ACTION[twisting_loop]
        
    def twist(self, element):
        res = self.F.one()
        for idx in element.Tietze():   # Tietze 表現を取得 (例: fg.x -> (1,), ~fg.x -> (-1,))
            c = self.GEN_NAMES[abs(idx)]
            image = self.act.get(c, self.F.gen(abs(idx) - 1))
            res *= image if idx > 0 else ~image
        return res

DT = DehnTwist

class MappingClass:
    def __init__(self, loops: str):
        self.loops = loops
        
    def __mul__(self, aMC):
        return type(self)(self.loops + aMC.loops)
        
    def inv(self):
        return type(self)(self.loops[::-1].swapcase())
        
    def conj(self, aMC):
        return aMC * self * aMC.inv()
#        return aMC.inv() * self * aMC
        
    def act_on_basis(self):
        twists = [DehnTwist(l) for l in self.loops[::-1]]
        return [
            reduce(lambda g, T: T.twist(g), twists, g)
            for g in fg.BASIS
        ]

MC = MappingClass

### Functions for the action on homology

In [3]:
# ============================================================
# H = π/[π,π] への射影
# ============================================================

# H_Z の基底順序
H_BASIS = ('X', 'Y', 'Z', 'W')

def homology(element):
    """
    FreeGroup F=<x,y,z,w> の元を

        H_Z = <X,Y,Z,W> ≅ Z^4

    に可換化する。

    座標順序:
        X, Y, Z, W
    """
    v = vector(ZZ, len(H_BASIS))

    for idx in element.Tietze():
        i = abs(idx) - 1
        v[i] += 1 if idx > 0 else -1

    return v

def matrix_from_columns(columns, ring=ZZ):
    """
    ベクトルのリスト columns を「列」とする行列を作る。
    """
    nrows = len(columns[0])
    ncols = len(columns)

    return matrix(
        ring,
        nrows,
        ncols,
        lambda i, j: columns[j][i]
    )

def homology_action_matrix(mc):
    return matrix_from_columns(
        [homology(g) for g in mc.act_on_basis()],
        ZZ
    )


def is_symplectic(A):
    J = matrix(ZZ, [
        [0,  1, 0,  0],
        [-1, 0, 0,  0],
        [0,  0, 0,  1],
        [0,  0, -1, 0]
    ])

    return A.transpose() * J * A == J    

### $\Lambda^{2}H$

In [4]:
# ============================================================
# Λ^2 H
# ============================================================

WEDGE_BASIS = (
    'X∧Y',
    'X∧Z',
    'X∧W',
    'Y∧Z',
    'Y∧W',
    'Z∧W'
)

# H の座標での index pair
WEDGE_PAIRS = (
    (0, 1),
    (0, 2),
    (0, 3),
    (1, 2),
    (1, 3),
    (2, 3)
)


def wedge(u, v):
    """
    u,v ∈ H に対して u∧v ∈ Λ^2 H を返す。

    Λ^2 H の基底順序:
        X∧Y, X∧Z, X∧W,
        Y∧Z, Y∧W, Z∧W
    """
    u = vector(QQ, u)
    v = vector(QQ, v)

    return vector(QQ, [
        u[i] * v[j] - u[j] * v[i]
        for i, j in WEDGE_PAIRS
    ])

### $\ell^{\theta}_{2}$

In [5]:
# ============================================================
# ell_2^theta の生成元での値
# ============================================================

def ell2_generators():
    return {
        1: vector(QQ, [ QQ(1)/2, 0, 0, 0, 0, 0]),  # x
        2: vector(QQ, [-QQ(1)/2, 0, 0, 0, 0, 0]),  # y
        3: vector(QQ, [0, 0, 0, 0, 0,  QQ(1)/2]),  # z
        4: vector(QQ, [0, 0, 0, 0, 0, -QQ(1)/2]),  # w
    }    
    
def ell2_letter(idx, ell2_generators):
    """
    Tietze の一文字 idx に対する ell_2^theta.

    idx:
         1 = x,  2 = y,  3 = z,  4 = w
        -1 = x^-1, ...

    ell_2(g^-1) = -ell_2(g)
    """
    value = vector(QQ, ell2_generators[abs(idx)])

    return value if idx > 0 else -value    

def ell2(element, ell2_generators):
    r"""
    FreeGroup の元 element に対して
        ell_2^theta(element) ∈ Λ^2 H
    を計算する。

    使用する公式:

        ell_2(uv)
          = ell_2(u)
          + ell_2(v)
          + 1/2 |u|∧|v|.
    """

    h = vector(QQ, 4)
    ell = vector(QQ, 6)

    for idx in element.Tietze():

        # 今読んでいる一文字の homology class
        v = vector(QQ, 4)

        i = abs(idx) - 1
        v[i] = 1 if idx > 0 else -1

        # product formula
        ell += (
            ell2_letter(idx, ell2_generators)
            + QQ(1)/2 * wedge(h, v)
        )

        # prefix の homology class を更新
        h += v

    return ell

### $N_{3}$

In [6]:
# ============================================================
# N_3 coordinate
# ============================================================

def N3_coordinate(element, ell2_generators):
    """
    [element]_3 を

        (ell_2^theta(element), |element|)

    として返す。
    """
    return (
        ell2(element, ell2_generators),
        vector(QQ, homology(element))
    )

def N3_multiply(a, b):
    r"""
    a = (xi, X)
    b = (nu, Y)

    に対して

        (xi,X)(nu,Y)
          =
        (xi + nu + 1/2 X∧Y,
         X + Y)
    """
    xi, X = a
    nu, Y = b

    X = vector(QQ, X)
    Y = vector(QQ, Y)

    return (
        vector(QQ, xi)
        + vector(QQ, nu)
        + QQ(1)/2 * wedge(X, Y),

        X + Y
    )


def N3_inverse(a):
    """
    (xi,X)^(-1) = (-xi,-X)
    """
    xi, X = a

    return (
        -vector(QQ, xi),
        -vector(QQ, X)
    )

# ============================================================
# Λ^2 A
# ============================================================

def wedge_action_matrix(A):
    """
    A : H -> H が Λ^2 H に誘導する行列。

        U∧V -> AU ∧ AV

    戻り値は 6×6 行列。
    """
    A = A.change_ring(QQ)

    columns = []

    for i, j in WEDGE_PAIRS:
        columns.append(
            wedge(A.column(i), A.column(j))
        )

    return matrix_from_columns(columns, QQ)

### $\tau^{\theta}_{1}(\phi)$

In [7]:
# ============================================================
# tau_1^theta
# ============================================================

def tau1_theta(mc, ell2_generators):
    """
    tau_1^theta(mc) : H -> Λ^2 H

    を 6×4 行列で返す。

    column order:
        X, Y, Z, W

    row order:
        X∧Y, X∧Z, X∧W,
        Y∧Z, Y∧W, Z∧W
    """

    basis = [fg.x, fg.y, fg.z, fg.w]

    images = mc.act_on_basis()

    A = homology_action_matrix(mc)
    A2 = wedge_action_matrix(A)

    delta_columns = []

    for g, image in zip(basis, images):

        delta = (
            ell2(image, ell2_generators)
            - A2 * ell2(g, ell2_generators)
        )

        delta_columns.append(delta)

    D = matrix_from_columns(delta_columns, QQ)

    A_QQ = A.change_ring(QQ)

    tau = D * A_QQ.inverse()

    return tau

### $\rho_{3}(\phi)$

In [8]:
# ============================================================
# rho_3
# ============================================================

def rho3(mc, ell2_generators):
    """
    rho_3(mc) を

        (tau_1^theta(mc), A)

    のデータとして返す。
    """

    A = homology_action_matrix(mc)

    return {
        'A': A,
        'A2': wedge_action_matrix(A),
        'tau': tau1_theta(mc, ell2_generators),
        'images': mc.act_on_basis()
    }

def rho3_apply(data, n3_element):
    """
    rho_3(phi) を N_3 の元に作用させる。

    data = rho3(phi, ...)
    n3_element = (xi, X)
    """
    xi, X = n3_element

    A = data['A'].change_ring(QQ)
    A2 = data['A2']
    tau = data['tau']

    X = vector(QQ, X)
    xi = vector(QQ, xi)

    AX = A * X

    return (
        tau * AX + A2 * xi,
        AX
    )

# Tests

### Tests for H-part

In [9]:
# ============================================================
# Tests for H-part
# convention:
#   H_BASIS = ('X','Y','Z','W')
#   i(X,Y)=1, i(Z,W)=1
# ============================================================


def test_homology_generators():
    """
    x,y,z,w とその逆元の可換化。
    座標順序は (X,Y,Z,W)。
    """
    assert homology(fg.x) == vector(ZZ, [1, 0, 0, 0])
    assert homology(fg.y) == vector(ZZ, [0, 1, 0, 0])
    assert homology(fg.z) == vector(ZZ, [0, 0, 1, 0])
    assert homology(fg.w) == vector(ZZ, [0, 0, 0, 1])

    assert homology(fg.X) == vector(ZZ, [-1, 0, 0, 0])
    assert homology(fg.Y) == vector(ZZ, [0, -1, 0, 0])
    assert homology(fg.Z) == vector(ZZ, [0, 0, -1, 0])
    assert homology(fg.W) == vector(ZZ, [0, 0, 0, -1])


def test_homology_homomorphism():
    """
    homology(uv)=homology(u)+homology(v)
    および逆元。
    """
    u = fg.x * fg.y * fg.Z
    v = fg.z * fg.w * fg.X

    assert homology(u * v) == homology(u) + homology(v)
    assert homology(~u) == -homology(u)


def test_commutators_vanish():
    """
    交換子は H 上で 0。
    """
    comm_xy = fg.x * fg.y * fg.X * fg.Y
    comm_zw = fg.z * fg.w * fg.Z * fg.W

    assert homology(comm_xy) == vector(ZZ, 4)
    assert homology(comm_zw) == vector(ZZ, 4)


def test_boundary_homology():
    """
    境界元は null-homologous。
    """
    assert homology(fg.bnd) == vector(ZZ, [0, 0, 0, 0])


def test_boundary_fixed_on_pi():
    """
    各 Dehn twist が境界元 bnd を π 上で固定することを確認。

    これは
        DehnTwist(generator) ∈ Aut_partial(pi)
    の重要な検算。
    """
    for loop in "aAbBcCdDeEfF":
        T = DehnTwist(loop)

        assert T.twist(fg.bnd) == fg.bnd


def test_inverse_twists_on_pi():
    """
    小文字 twist と大文字 twist が π 上で互いに逆であること。
    """
    for lower, upper in [
        ('a', 'A'),
        ('b', 'B'),
        ('c', 'C'),
        ('d', 'D'),
        ('e', 'E'),
        ('f', 'F'),
    ]:
        identity1 = MC(lower + upper)
        identity2 = MC(upper + lower)

        assert identity1.act_on_basis() == [
            fg.x, fg.y, fg.z, fg.w
        ]

        assert identity2.act_on_basis() == [
            fg.x, fg.y, fg.z, fg.w
        ]


def test_identity_mapping_class():
    """
    MC("") は恒等写像。
    """
    identity = MC("")

    assert identity.act_on_basis() == [
        fg.x, fg.y, fg.z, fg.w
    ]

    assert homology_action_matrix(identity) == identity_matrix(ZZ, 4)


def test_matrix_from_columns():
    """
    matrix_from_columns が列 convention であること。
    """
    e1 = vector(ZZ, [1, 0, 0, 0])
    e2 = vector(ZZ, [0, 1, 0, 0])
    e3 = vector(ZZ, [0, 0, 1, 0])
    e4 = vector(ZZ, [0, 0, 0, 1])

    A = matrix_from_columns([e1, e2, e3, e4], ZZ)

    assert A == identity_matrix(ZZ, 4)


def test_dehn_twist_a():
    """
    a:
        y -> x y

    よって H 上では
        Y -> X + Y.
    """
    A = homology_action_matrix(MC("a"))

    expected = matrix(ZZ, [
        [1, 1, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ])

    assert A == expected


def test_dehn_twist_b():
    """
    b:
        x -> w y^{-1} x
        z -> z y w^{-1}

    よって
        X -> X - Y + W
        Z -> Y + Z - W.
    """
    A = homology_action_matrix(MC("b"))

    expected = matrix(ZZ, [
        [ 1, 0, 0, 0],
        [-1, 1, 1, 0],
        [ 0, 0, 1, 0],
        [ 1, 0,-1, 1],
    ])

    assert A == expected


def test_dehn_twist_c():
    """
    c:
        w -> z w

    よって
        W -> Z + W.
    """
    A = homology_action_matrix(MC("c"))

    expected = matrix(ZZ, [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 1],
        [0, 0, 0, 1],
    ])

    assert A == expected


def test_dehn_twist_d():
    """
    d:
        z -> bnd^{-1} w^{-1} z

    |bnd|=0 なので
        Z -> Z - W.
    """
    A = homology_action_matrix(MC("d"))

    expected = matrix(ZZ, [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0,-1, 1],
    ])

    assert A == expected


def test_dehn_twist_e():
    """
    e:
        x -> w x w^{-1}
        y -> w y w^{-1}
        z -> z w^{-1}

    共役部分は H 上で消えるので
        X -> X
        Y -> Y
        Z -> Z - W.
    """
    A = homology_action_matrix(MC("e"))

    expected = matrix(ZZ, [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0,-1, 1],
    ])

    assert A == expected


def test_dehn_twist_f():
    """
    f:
        x -> x y^{-1}

    よって
        X -> X - Y.
    """
    A = homology_action_matrix(MC("f"))

    expected = matrix(ZZ, [
        [ 1, 0, 0, 0],
        [-1, 1, 0, 0],
        [ 0, 0, 1, 0],
        [ 0, 0, 0, 1],
    ])

    assert A == expected


def test_inverse_twists_on_H():
    """
    大文字が H 上でも小文字の逆行列になっているか。
    """
    for lower, upper in [
        ('a', 'A'),
        ('b', 'B'),
        ('c', 'C'),
        ('d', 'D'),
        ('e', 'E'),
        ('f', 'F'),
    ]:
        A = homology_action_matrix(MC(lower))
        Ainv = homology_action_matrix(MC(upper))

        assert A * Ainv == identity_matrix(ZZ, 4)
        assert Ainv * A == identity_matrix(ZZ, 4)


def test_symplectic_generators():
    """
    各 Dehn twist が H 上で symplectic であること。
    """
    for loop in "aAbBcCdDeEfF":
        A = homology_action_matrix(MC(loop))

        assert A.det() == 1
        assert is_symplectic(A)


def test_composition_order():
    """
    現在の MappingClass の convention:

        mc1 * mc2

    は写像として
        mc1 ∘ mc2

    になる。

    よって
        A(mc1 * mc2) = A(mc1) A(mc2).
    """
    mc1 = MC("Db")
    mc2 = MC("Ca")

    lhs = homology_action_matrix(mc1 * mc2)

    rhs = (
        homology_action_matrix(mc1)
        * homology_action_matrix(mc2)
    )

    assert lhs == rhs


def test_DbCa_regression():
    """
    今回の具体例:
        phi = D b C a
    """

    phi = MC("DbCa")

    # --- π 上の像 ---
    expected_images = [
        fg.x * fg.Y * fg.z * fg.w * fg.Z,

        fg.z * fg.W * fg.Z
            * fg.y * fg.z * fg.w * fg.Z
            * fg.x * fg.Y * fg.z * fg.w * fg.Z,

        fg.z * fg.W * fg.Z
            * fg.y * fg.z * fg.w,

        fg.Z * fg.Y * fg.z * fg.w * fg.Z,
    ]

    assert phi.act_on_basis() == expected_images

    # --- H 上の作用 ---
    A = homology_action_matrix(phi)

    expected_A = matrix(ZZ, [
        [ 1, 1, 0,  0],
        [-1, 0, 1, -1],
        [ 0, 0, 1, -1],
        [ 1, 1, 0,  1],
    ])

    assert A == expected_A

    assert A.det() == 1
    assert is_symplectic(A)


def run_H_tests():
    tests = [
        test_homology_generators,
        test_homology_homomorphism,
        test_commutators_vanish,
        test_boundary_homology,
        test_boundary_fixed_on_pi,
        test_inverse_twists_on_pi,
        test_identity_mapping_class,
        test_matrix_from_columns,
        test_dehn_twist_a,
        test_dehn_twist_b,
        test_dehn_twist_c,
        test_dehn_twist_d,
        test_dehn_twist_e,
        test_dehn_twist_f,
        test_inverse_twists_on_H,
        test_symplectic_generators,
        test_composition_order,
        test_DbCa_regression,
    ]

    for test in tests:
        test()
        print(f"✓ {test.__name__}")

    print()
    print(f"All {len(tests)} H-tests passed.")

In [10]:
run_H_tests()

✓ test_homology_generators
✓ test_homology_homomorphism
✓ test_commutators_vanish
✓ test_boundary_homology
✓ test_boundary_fixed_on_pi
✓ test_inverse_twists_on_pi
✓ test_identity_mapping_class
✓ test_matrix_from_columns
✓ test_dehn_twist_a
✓ test_dehn_twist_b
✓ test_dehn_twist_c
✓ test_dehn_twist_d
✓ test_dehn_twist_e
✓ test_dehn_twist_f
✓ test_inverse_twists_on_H
✓ test_symplectic_generators
✓ test_composition_order
✓ test_DbCa_regression

All 18 H-tests passed.


### Tests for DehnTwists and MappingClass

In [11]:
# ============================================================
# Tests for DehnTwists and MappingClass
# ============================================================


def test_dehn_twist_single_action():
    """
    DehnTwist.twist() の基本的な置換作用を確認。
    """

    a = DT('a')
    A = DT('A')

    assert a.twist(fg.x) == fg.x
    assert a.twist(fg.y) == fg.y * fg.x

    assert A.twist(fg.x) == fg.x
    assert A.twist(fg.y) == fg.y * fg.X


def test_dehn_twist_on_word():
    """
    twist が自由群の語に対して準同型として作用することを確認。

        T(uv) = T(u)T(v)
    """

    T = DT('b')

    u = fg.x * fg.Y * fg.z
    v = fg.w * fg.x

    assert T.twist(u * v) == T.twist(u) * T.twist(v)


def test_dehn_twist_inverse_on_basis():
    """
    小文字と対応する大文字が互いに逆写像であることを
    x,y,z,w 全てについて確認。
    """

    for lower, upper in zip(
        "abcdef",
        "ABCDEF"
    ):
        T = DT(lower)
        T_inv = DT(upper)

        for g in fg.BASIS:

            assert T_inv.twist(T.twist(g)) == g
            assert T.twist(T_inv.twist(g)) == g


def test_dehn_twist_fixes_boundary():
    """
    全ての Dehn twist generator が境界語 bnd を固定すること。

        T(bnd) = bnd
    """

    for letter in "aAbBcCdDeEfF":

        T = DT(letter)

        assert T.twist(fg.bnd) == fg.bnd


def test_mappingclass_single_twist():
    """
    MC("a") の作用と DT("a") の作用が一致すること。
    """

    mc = MC("a")
    T = DT("a")

    assert mc.act_on_basis() == [
        T.twist(g)
        for g in fg.BASIS
    ]


def test_mappingclass_composition():
    r"""
    MappingClass の積 convention を確認。

        MC("a") * MC("b")

    は作用として

        a o b

    すなわち最初に b、その後 a を作用させる。
    """

    a = DT("a")
    b = DT("b")

    mc = MC("a") * MC("b")

    expected = [
        a.twist(
            b.twist(g)
        )
        for g in fg.BASIS
    ]

    assert mc.act_on_basis() == expected


def test_mappingclass_inverse():
    """
    mc.inv() が mapping class の逆元を与えること。
    """

    mc = MC("DbCa")

    identity1 = mc * mc.inv()
    identity2 = mc.inv() * mc

    assert identity1.act_on_basis() == list(fg.BASIS)
    assert identity2.act_on_basis() == list(fg.BASIS)


def test_mappingclass_conjugation():
    r"""
    現在の convention

        phi.conj(h) = h phi h^{-1}

    が homology representation と一致することを確認。

        |h phi h^{-1}|
          =
        |h| |phi| |h|^{-1}.
    """

    phi = MC("a")
    h = MC("b")

    A_phi = homology_action_matrix(phi)
    A_h = homology_action_matrix(h)

    A_conj = homology_action_matrix(
        phi.conj(h)
    )

    assert (
        A_conj
        == A_h * A_phi * A_h.inverse()
    )


def test_mappingclass_conjugation_on_pi():
    r"""
    conjugation convention を pi 上でも直接確認。

        phi.conj(h)
          =
        h phi h^{-1}
    """

    phi = MC("a")
    h = MC("b")

    conj_mc = phi.conj(h)

    # h phi h^{-1} を明示的に作る
    explicit = h * phi * h.inv()

    assert (
        conj_mc.act_on_basis()
        == explicit.act_on_basis()
    )


# ============================================================
# Run all DehnTwist / MappingClass tests
# ============================================================

def run_DehnTwist_tests():

    tests = [
        test_dehn_twist_single_action,
        test_dehn_twist_on_word,
        test_dehn_twist_inverse_on_basis,
        test_dehn_twist_fixes_boundary,

        test_mappingclass_single_twist,
        test_mappingclass_composition,
        test_mappingclass_inverse,
        test_mappingclass_conjugation,
        test_mappingclass_conjugation_on_pi,
    ]

    for test in tests:
        test()
        print(f"✓ {test.__name__}")

    print()
    print(
        f"All {len(tests)} DehnTwist/MappingClass tests passed."
    )

In [12]:
run_DehnTwist_tests()

✓ test_dehn_twist_single_action
✓ test_dehn_twist_on_word
✓ test_dehn_twist_inverse_on_basis
✓ test_dehn_twist_fixes_boundary
✓ test_mappingclass_single_twist
✓ test_mappingclass_composition
✓ test_mappingclass_inverse
✓ test_mappingclass_conjugation
✓ test_mappingclass_conjugation_on_pi

All 9 DehnTwist/MappingClass tests passed.


### Tests for $N_3$

In [23]:
# ============================================================
# Tests for N_3-part
#
# convention:
#   H basis:
#       X, Y, Z, W
#
#   Lambda^2 H basis:
#       X∧Y, X∧Z, X∧W,
#       Y∧Z, Y∧W, Z∧W
#
#   [g]_3 <-> (ell(g), |g|)
# ============================================================


# 今回使う ell の生成元での値
ELL = ell2_generators()


def test_ell_generators():
    """
    ell の生成元での値を確認。
    """

    assert ell2(fg.x, ELL) == vector(QQ, [
         QQ(1)/2, 0, 0, 0, 0, 0
    ])

    assert ell2(fg.y, ELL) == vector(QQ, [
        -QQ(1)/2, 0, 0, 0, 0, 0
    ])

    assert ell2(fg.z, ELL) == vector(QQ, [
        0, 0, 0, 0, 0, QQ(1)/2
    ])

    assert ell2(fg.w, ELL) == vector(QQ, [
        0, 0, 0, 0, 0, -QQ(1)/2
    ])


def test_ell_inverse():
    """
    ell(g^{-1}) = -ell(g)
    を一般の語で確認。
    """

    g = fg.x * fg.Y * fg.z * fg.w * fg.X

    assert ell2(~g, ELL) == -ell2(g, ELL)


def test_ell_product_formula():
    r"""
    ell(uv)
      =
    ell(u) + ell(v) + 1/2 |u|∧|v|
    を確認。
    """

    u = fg.x * fg.Y * fg.z
    v = fg.w * fg.x * fg.Z

    lhs = ell2(u * v, ELL)

    rhs = (
        ell2(u, ELL)
        + ell2(v, ELL)
        + QQ(1)/2 * wedge(
            homology(u),
            homology(v)
        )
    )

    assert lhs == rhs


def test_boundary_ell():
    """
    境界元について

        ell(bnd) = omega
                 = X∧Y + Z∧W

    を確認。
    """

    omega = vector(QQ, [
        1, 0, 0, 0, 0, 1
    ])

    assert ell2(fg.bnd, ELL) == omega


# ------------------------------------------------------------
# N_3 の群構造
# ------------------------------------------------------------

def test_N3_identity():
    """
    単位元 1 ∈ pi が
        (0,0) ∈ Lambda^2 H ⋊ H
    に対応すること。
    """

    one = fg.F.one()

    expected = (
        vector(QQ, 6),
        vector(QQ, 4)
    )

    assert N3_coordinate(one, ELL) == expected


def test_N3_product():
    """
    座標写像が積を保つこと:

        coord(uv)
          =
        coord(u) coord(v).
    """

    u = fg.x * fg.Y * fg.z
    v = fg.w * fg.x * fg.Z * fg.y

    lhs = N3_coordinate(u * v, ELL)

    rhs = N3_multiply(
        N3_coordinate(u, ELL),
        N3_coordinate(v, ELL)
    )

    assert lhs == rhs


def test_N3_inverse():
    """
    coord(g^{-1})
      =
    coord(g)^{-1}.
    """

    g = (
        fg.x * fg.Y * fg.z
        * fg.w * fg.X * fg.y
    )

    lhs = N3_coordinate(~g, ELL)

    rhs = N3_inverse(
        N3_coordinate(g, ELL)
    )

    assert lhs == rhs


def test_N3_associativity():
    """
    実装した N3_multiply が結合的であること。
    """

    a = N3_coordinate(
        fg.x * fg.y,
        ELL
    )

    b = N3_coordinate(
        fg.Z * fg.w * fg.x,
        ELL
    )

    c = N3_coordinate(
        fg.y * fg.W,
        ELL
    )

    lhs = N3_multiply(
        N3_multiply(a, b),
        c
    )

    rhs = N3_multiply(
        a,
        N3_multiply(b, c)
    )

    assert lhs == rhs


# ------------------------------------------------------------
# Lambda^2 A
# ------------------------------------------------------------

def test_wedge_action_matrix():
    r"""
    Lambda^2 A が本当に

        (Lambda^2 A)(U∧V)
          =
        AU ∧ AV

    を満たすこと。
    """

    phi = MC("DbCa")

    A = homology_action_matrix(phi).change_ring(QQ)
    A2 = wedge_action_matrix(A)

    U = vector(QQ, [1, 2, -1, 1])
    V = vector(QQ, [-1, 1, 2, 0])

    lhs = A2 * wedge(U, V)

    rhs = wedge(
        A * U,
        A * V
    )

    assert lhs == rhs


# ------------------------------------------------------------
# tau_1^theta
# ------------------------------------------------------------

def test_tau_defining_relation():
    r"""
    各基底 g=x,y,z,w について

        ell(phi(g))
          =
        tau(A|g|)
        + (Lambda^2 A) ell(g)

    を確認する。
    """

    phi = MC("DbCa")

    A = homology_action_matrix(phi).change_ring(QQ)
    A2 = wedge_action_matrix(A)
    tau = tau1_theta(phi, ELL)

    basis = fg.BASIS
    images = phi.act_on_basis()

    for g, image in zip(basis, images):

        X = vector(QQ, homology(g))

        lhs = ell2(
            image,
            ELL
        )

        rhs = (
            tau * (A * X)
            + A2 * ell2(g, ELL)
        )

        assert lhs == rhs


# ------------------------------------------------------------
# rho_3
# ------------------------------------------------------------

def test_rho3_identity():
    """
    恒等写像について

        A = I,
        tau = 0

    であること。
    """

    identity = MC("")

    data = rho3(identity, ELL)

    assert data['A'] == identity_matrix(ZZ, 4)
    assert data['A2'] == identity_matrix(QQ, 6)
    assert data['tau'] == zero_matrix(QQ, 6, 4)


def test_rho3_DbCa_regression():
    """
    phi = DbCa について、
    現在確定している A と tau の値を確認。
    """

    phi = MC("DbCa")

    data = rho3(phi, ELL)

    expected_A = matrix(ZZ, [
        [ 1, 1, 0,  0],
        [-1, 0, 1, -1],
        [ 0, 0, 1, -1],
        [ 1, 1, 0,  1],
    ])

    expected_tau = (1/2) * matrix(QQ, [
        [ 0,  0,  1,  0],
        [ 0,  0,  0,  0],
        [-1,  0,  0,  0],
        [ 0,  0, -1,  0],
        [ 0, -1,  0, -1],
        [-1,  0,  0,  0],
    ])

    assert data['A'] == expected_A
    assert data['tau'] == expected_tau


def test_rho3_on_basis():
    r"""
    phi = DbCa について

        rho_3(phi)(coord(g))
          =
        coord(phi(g))

    を g=x,y,z,w で確認。
    """

    phi = MC("DbCa")

    data = rho3(phi, ELL)

    for g, image in zip(
        fg.BASIS,
        phi.act_on_basis()
    ):

        lhs = rho3_apply(
            data,
            N3_coordinate(g, ELL)
        )

        rhs = N3_coordinate(
            image,
            ELL
        )

        assert lhs == rhs


def test_rho3_fixes_boundary():
    r"""
    Aut_partial(pi) の元は bnd を固定するので、
    N_3 上でも bnd の座標を固定することを確認。

    Humphries generators と DbCa の両方をテストする。
    """

    boundary_coord = N3_coordinate(
        fg.bnd,
        ELL
    )

    for loops in [
        "a", "A",
        "b", "B",
        "c", "C",
        "d", "D",
        "e", "E",
        "f", "F",
        "DbCa",
    ]:

        data = rho3(
            MC(loops),
            ELL
        )

        assert rho3_apply(
            data,
            boundary_coord
        ) == boundary_coord


def test_rho3_composition():
    r"""
    MappingClass の積 convention

        mc1 * mc2 = mc1 o mc2

    と rho_3 の半直積の積が一致することを確認。

    tau の積公式は

        tau(mc1 mc2)
          =
        tau1
        + (Lambda^2 A1) tau2 A1^{-1}.
    """

    mc1 = MC("Db")
    mc2 = MC("Ca")

    R1 = rho3(mc1, ELL)
    R2 = rho3(mc2, ELL)
    R12 = rho3(mc1 * mc2, ELL)

    A1 = R1['A'].change_ring(QQ)
    A2 = R2['A'].change_ring(QQ)

    # H-part
    assert (
        R12['A'].change_ring(QQ)
        == A1 * A2
    )

    # Lambda^2 H-part
    assert (
        R12['A2']
        == R1['A2'] * R2['A2']
    )

    # Johnson part
    expected_tau = (
        R1['tau']
        + R1['A2']
          * R2['tau']
          * A1.inverse()
    )

    assert R12['tau'] == expected_tau


# ============================================================
# Run all N_3 tests
# ============================================================

def run_N3_tests():

    tests = [
        # ell
        test_ell_generators,
        test_ell_inverse,
        test_ell_product_formula,
        test_boundary_ell,

        # N_3 group law
        test_N3_identity,
        test_N3_product,
        test_N3_inverse,
        test_N3_associativity,

        # Lambda^2 A
        test_wedge_action_matrix,

        # tau
        test_tau_defining_relation,

        # rho_3
        test_rho3_identity,
#        test_rho3_DbCa_regression,
        test_rho3_on_basis,
        test_rho3_fixes_boundary,
        test_rho3_composition,
    ]

    for test in tests:
        test()
        print(f"✓ {test.__name__}")

    print()
    print(f"All {len(tests)} N_3-tests passed.")

In [24]:
run_N3_tests()

✓ test_ell_generators
✓ test_ell_inverse
✓ test_ell_product_formula
✓ test_boundary_ell
✓ test_N3_identity
✓ test_N3_product
✓ test_N3_inverse
✓ test_N3_associativity
✓ test_wedge_action_matrix
✓ test_tau_defining_relation
✓ test_rho3_identity
✓ test_rho3_on_basis
✓ test_rho3_fixes_boundary
✓ test_rho3_composition

All 14 N_3-tests passed.


# Scratch